In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq          # row counts / column names without loading files
from pathlib import Path
from IPython.display import display  

THis is tfafaw

In [2]:
# Cell 1
import pandas as pd, numpy as np
import pyarrow.parquet as pq
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'evaluation.py').exists())
path = ROOT / 'data'
files = sorted(path.glob('*.parquet'))
files

[PosixPath('../data/company_years.parquet'),
 PosixPath('../data/company_years_h1.parquet'),
 PosixPath('../data/company_years_h2.parquet'),
 PosixPath('../data/company_years_h3.parquet'),
 PosixPath('../data/company_years_h4.parquet'),
 PosixPath('../data/company_years_h5.parquet'),
 PosixPath('../data/company_years_h6.parquet')]

In [3]:
# Cell 2: rows and columns without loading the data
rows = []
for f in files:
    meta = pq.ParquetFile(f).metadata
    rows.append({'file': f.name, 'rows': meta.num_rows,
                 'cols': meta.num_columns, 'MB': round(f.stat().st_size/1e6)})
pd.DataFrame(rows)

,file,rows,cols,MB
0,company_years.parquet,1106879,142,894
1,company_years_h1.parquet,1000087,143,823
2,company_years_h2.parquet,996500,143,820
3,company_years_h3.parquet,898692,143,742
4,company_years_h4.parquet,793234,143,655
5,company_years_h5.parquet,700041,143,577
6,company_years_h6.parquet,598832,143,501


In [4]:
df = pd.read_parquet(path/'company_years_h2.parquet')   # 1-year-ahead file
df.shape

(996500, 143)

In [5]:
DROP = ['company', 'industry', 'link', 'num', 'emis_id', 'sector_2', 'sector_3', 'sector_4',
        'Revenue/employee', 'Fixed_assets/employee', 'EBITDA/cash_flow']
[c for c in DROP if c not in df.columns]   # should print []

[]

In [6]:
df = pd.read_parquet(path/'company_years_h2.parquet')     # 1-year-ahead file
df.shape

(996500, 143)

In [ ]:
display(df.head(3).T)                                   # transposed so all 143 columns fit
display(df.dtypes.value_counts())
print(df.select_dtypes(exclude='number').columns.tolist())
display(df[['country', 'legal_form', 'operational_status']].nunique())

In [ ]:
cid = df.loc[df.main_label == 1, 'company'].iloc[0]
print('company:', cid)

for h in range(1, 7):
    f = path/f'company_years_h{h}.parquet'
    d = pd.read_parquet(f, columns=['company', 'year', 'main_label'])
    rows = d[d.company == cid].sort_values('year')
    print(f"\n{f.name}  (horizon {h-1} = distress {h-1} yr later)")
    print(rows[['year', 'main_label']].to_string(index=False))

In [ ]:
pd.set_option('display.max_columns', None)   # stop hiding columns behind "..."
pd.set_option('display.width', None)
df.head(10)

In [7]:
miss = df.isna().mean().sort_values(ascending=False)
print(f"columns with any missing: {(miss > 0).sum()},  >50% missing: {(miss > 0.5).sum()}")
display(miss.head(15))

# is missingness different for distressed firms?
top = miss.index[:8]
display(df.groupby('main_label')[top].apply(lambda g: g.isna().mean()).T)

columns with any missing: 121,  >50% missing: 9


Fixed_assets/employee                             1.000000
Revenue/employee                                  1.000000
sector_4                                          0.999830
sector_3                                          0.996927
sector_2                                          0.969828
EBITDA/cash_flow                                  0.958354
ST_financial_assets/current_liabilities_Sector    0.864565
Cash_flow/sales                                   0.589203
Cash_flow/total_debt                              0.570151
Equity/long_term_liabilities                      0.478187
Fixed_assets/long_term_liabilities                0.478187
Revenue/long_term_liabilities                     0.478187
Inventories_growth                                0.426536
Operating_profit_growth                           0.316744
Operating_revenue_growth                          0.301081
dtype: float64

main_label,0,1
Fixed_assets/employee,1.000000,1.000000
Revenue/employee,1.000000,1.000000
sector_4,0.999830,1.000000
sector_3,0.996924,0.998035
sector_2,0.969770,0.988867
EBITDA/cash_flow,0.958354,0.958415
ST_financial_assets/current_liabilities_Sector,0.864362,0.930583
Cash_flow/sales,0.589243,0.576293


In [8]:
num = df.select_dtypes('number')
inf = np.isinf(num).sum()
display(inf[inf > 0].sort_values(ascending=False).head(10))

Series([], dtype: int64)

In [9]:
key = ['Equity/total_assets', 'EBITDA/total_assets', 'Current_assets/short_term_liabilities',
       'Net_profit/total_assets', 'Working_capital/total_assets', 'Total_liabilities/total_assets']
display(df[key].replace([np.inf, -np.inf], np.nan)
          .describe(percentiles=[.01, .5, .99]).T)
display(df.groupby('main_label')[key].median().T)

,count,mean,std,min,1%,50%,99%,max
Equity/total_assets,996236.0,-0.037504,37.886063,-24707.876712,-3.896079,0.420958,1.000000,4.980645e+02
EBITDA/total_assets,996236.0,0.088149,11.551771,-11070.329971,-0.646260,0.074697,0.785168,1.501454e+03
Current_assets/short_term_liabilities,963504.0,17.822305,1632.625314,-46.683422,0.084867,1.498110,106.540859,1.148428e+06
Net_profit/total_assets,996236.0,0.017558,7.854869,-4295.203667,-1.267363,0.040300,0.775661,2.718590e+03
Working_capital/total_assets,996236.0,-0.093600,36.322078,-24707.876712,-2.857246,0.241757,1.000000,4.980645e+02
Total_liabilities/total_assets,996236.0,1.016673,37.834585,-497.064516,0.000000,0.570531,4.700038,2.470888e+04


main_label,0,1
Equity/total_assets,0.422280,-0.306962
EBITDA/total_assets,0.075008,-0.074806
Current_assets/short_term_liabilities,1.501725,0.553747
Net_profit/total_assets,0.040571,-0.168114
Working_capital/total_assets,0.242888,-0.496503
Total_liabilities/total_assets,0.569120,1.305531


In [10]:
display(pd.crosstab(df.operational_status, df.main_label, normalize='index'))
display(df.groupby('year').main_label.agg(['sum', 'mean']))

main_label,0,1
operational_status,,
0.0,0.988889,0.011111
1.0,0.998026,0.001974


,sum,mean
year,,
2006,21,0.000814
2007,52,0.001369
2008,99,0.001988
2009,162,0.002948
2010,146,0.002579
2011,176,0.002880
2012,180,0.002773
2013,220,0.003333
2014,257,0.003738


In [11]:
display(df.operational_status.value_counts())
print(df.groupby('company').operational_status.nunique().max())   # 1 = never changes within a company

operational_status
1.0    877523
0.0    118977
Name: count, dtype: int64

2


In [12]:
NOT_FEATURES = ['operational_status', 'year', 'main_label']

In [13]:
print(df.groupby('company').operational_status.nunique().max())

2


In [14]:
# 1. How many companies ever change status?
n_status = df.groupby('company').operational_status.nunique()
print(f"companies with a status change: {(n_status > 1).sum():,} of {len(n_status):,} "
      f"({(n_status > 1).mean():.2%})")

# 2. For distressed firms: is the status different in the labelled year than in earlier years?
pos_firms = df.loc[df.main_label == 1, 'company'].unique()
sub = df[df.company.isin(pos_firms)]
display(sub.groupby('main_label').operational_status.value_counts(normalize=True).unstack())

companies with a status change: 209 of 187,525 (0.11%)


operational_status,0.0,1.0
main_label,,
0,0.456759,0.543241
1,0.432875,0.567125
